In [1]:
import os
import h5py
import numpy as np
import pandas as pd

In [ ]:
# Read in labeled folder names and timestamps
Cluster_detail_results = pd.read_csv( os.path.join(r'X:\Members\Mia-Sanjana-Hadent\Processed Data\042025_2mp\Cluster_detail_results.csv') )

In [ ]:
# CHANGE HERE: path to your mat file
filename = r"X:\Members\Mia-Sanjana-Hadent\Processed Data\042025_2mp\session_1_out.mat"

In [ ]:
# Read in values from clustersIdx, set automatically to the cluster value it represents rather than the actual value of the cell
if os.path.exists(filename):
    with h5py.File(filename, 'r') as file:
        library = file['Library']
        clustersIdx = library['clustersIdx']
        
        group_labels = []
        for i in range(clustersIdx.shape[0]):
            idxref = clustersIdx[i, 0]
            cluster_indices = np.array(file[idxref]).flatten()
            group_labels.extend([i+1] * len(cluster_indices))
        
        group_labels_df = pd.DataFrame(group_labels, columns=['GroupLabel'])
        print(f"Group labels shape: {group_labels_df.shape}")
else:
    raise FileNotFoundError('Could not find session_1_out.mat in dedicated subdirectory')


group_labels_df

In [ ]:
# Read in the feature values from clusters and stack on top of each other
if os.path.exists(filename):
    print('Loading session file from expected Results/test1 directory')
    with h5py.File(filename, 'r') as file:
        if 'Library' not in file:
            raise KeyError("The key 'Library' was not found in the session file.")
        library = file['Library']
        clusters = library['clusters']
        
        features_data = []
        
        for i in range(clusters.shape[0]):
            cref = clusters[i, 0]
            feature_data = np.array(file[cref]).T 
            features_data.append(feature_data)
        
        features_matrix = np.vstack(features_data)
        features_df = pd.DataFrame(features_matrix)
        print(f"Loaded {len(features_data)} clusters with total points: {features_matrix.shape[0]}")
        print(f"Combined matrix shape: {features_matrix.shape}")
else:
    raise FileNotFoundError('Could not find session_1_out.mat in dedicated subdirectory')
features_df['cluster'] = group_labels_df['GroupLabel'].values


features_df

In [ ]:
# Create combined matrix by matching first occurrence of each cluster in features_df to first occurrence in Cluster_detail_results, keeping order of Cluster_detail_results
def match_features_to_details(features_df, detail_df):
    matched_rows = []
    cluster_counts = {}
    for idx, detail_row in detail_df.iterrows():
        cluster_val = detail_row['ClusterIdx']
        count = cluster_counts.get(cluster_val, 0)
        feature_rows = features_df[features_df['cluster'] == cluster_val]
        if count < len(feature_rows):
            feature_row = feature_rows.iloc[count]
            combined_row = feature_row.to_dict()
            combined_row['Timestamp'] = detail_row['Timestamp'] if 'Timestamp' in detail_row else None
            combined_row['ClusterIdx'] = cluster_val
            matched_rows.append(combined_row)
            cluster_counts[cluster_val] = count + 1
        else:
            combined_row = {col: None for col in features_df.columns}
            combined_row['Timestamp'] = detail_row['Timestamp'] if 'Timestamp' in detail_row else None
            combined_row['ClusterIdx'] = cluster_val
            matched_rows.append(combined_row)
    return pd.DataFrame(matched_rows)

In [ ]:
combined_matrix = match_features_to_details(features_df, Cluster_detail_results)
combined_matrix

In [ ]:
# Clean up the finalized combined df (Column Titles : "Feature 1" ... "Feature 30" , "Timestamp" , "Cluster")
feature_cols = {i: f'Feature{i+1}' for i in range(30)}
combined_matrix = combined_matrix.rename(columns=feature_cols)
combined_matrix = combined_matrix.drop(columns=['cluster'])
combined_matrix = combined_matrix.rename(columns={'ClusterIdx': 'Cluster'})
combined_matrix

In [ ]:
# curr_dir = os.getcwd()
# base_dir = os.path.dirname(curr_dir)
# arena_data_dir = os.path.join(base_dir, '3D_Data')
# arena_visualizations_dir = os.path.join( base_dir , '3D_Visualizations')

In [ ]:
# if os.path.exists(filename):
#     print('Loading session file from expected Results/test1 directory')
#     with h5py.File(filename, 'r') as file:
#         if 'Library' not in file:
#             raise KeyError("The key 'Library' was not found in the session file.")
#         library = file['Library']
#         clusters = library['clusters']
#         clustersIdx = library['clustersIdx']

#         combined_data = []

#         for i in range(clusters.shape[0]):
#             cref = clusters[i, 0]
#             idxref = clustersIdx[i, 0]

#             cluster_data = np.array(file[cref]).T
#             cluster_indices = np.array(file[idxref]).flatten()

#             cluster_number_col = np.full((cluster_data.shape[0], 1), i+1, dtype=int)

#             combined = np.hstack((cluster_data, 
#                                   cluster_indices[:, None],
#                                   cluster_number_col))

#             combined_data.append(combined)

#         combined_matrix = np.vstack(combined_data)

#         print(f"Loaded {len(combined_data)} clusters with total points: {combined_matrix.shape[0]}")
#         print(f"Combined matrix shape: {combined_matrix.shape}")

# else:
#     raise FileNotFoundError('Could not find session_1_out.mat in dedicated subdirectory')


In [ ]:
# CHANGE HERE: File name

# output_file = os.path.join(arena_data_dir, 'combined_matrix_final_total136_lc.csv')
# df.to_csv(output_file, index=False)
# print(f"Saved as {output_file}")
# print(df.head())